# CS-781 Phase 5 (Legal Statute Identification) - data inspection

Run all (CPU is enough, a few minutes; Kaggle with Internet on and secrets `HF`, `HF_WRITE`). Reads `Phase_5_LSI/Train/train.json`, `Train/dev.json`, `Test/test.json` and prints: fields, id/text overlaps between train, dev and test, label vocabulary and support (how many rare statutes), labels per document, label co-occurrence, document lengths in Phase-1 BPE tokens, whether 'Section N' / '... Act' mentions predict the labels and where in the document they sit, and trivial baselines on dev. Send the whole printout back: Phase 5 settings (document head, loss, chunking) are fixed from it.

In [1]:
TOKENIZER_NAME = "a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json"
!pip install -q -U "huggingface_hub>=1.5"
import os, json, shutil, subprocess, sys, time, zipfile, random, math, gc
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download, login, download_bucket_files, batch_bucket_files
from kaggle_secrets import UserSecretsClient

NB_START = time.time()
WORK = "/kaggle/working"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # old repo, read-only fallback
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def get_dataset_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)


def fetch_model_file(name, local_path=None):
    local_path = local_path or f"{WORK}/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    # fallback: the OLD model repo (the Phase-1 notebooks do the same; the tokenizer only lives there)
    p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
    shutil.copy(p, local_path)
    print(f"(found {name} in the old repo {HF_CHECKPOINT_REPO})")
    return local_path


def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    try:
        batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
        print(f"uploaded {name} to bucket {HF_BUCKET}")
    except Exception as e:
        print(f"(upload of {name} failed: {type(e).__name__}: {e}; the file is still in {WORK})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 855.2/855.2 kB 16.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.3.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.3.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.3.0 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.3.0 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.3.0 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.3.0 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cpu 


In [2]:
import array, heapq, time
from collections import Counter, defaultdict

BPE_MIN_WORD_FREQ = 1      # 1 = learn merges from every word type (exactly the Assignment-1 algorithm)
_END = "</w>"


class BPETokenizer:
    """Byte-level BPE (Assignment 1). Initial symbols are the UTF-8 bytes of each word, held as chr(b), plus
    '</w>' marking the end of a word so merges never cross word boundaries. Ties in pair frequency break
    lexicographically. Text is NOT lowercased."""

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.merges = []
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {}
        self._ranks = {}
        self._id_cache = {}

    # ---- helpers ----
    def _word_symbols(self, word):
        syms = [chr(b) for b in word.encode("utf-8")]
        syms.append(_END)
        return syms

    def _merge(self, symbols, pair):
        res, i = [], 0
        a, b = pair
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                res.append(a + b)
                i += 2
            else:
                res.append(symbols[i])
                i += 1
        return res

    # ---- training ----
    def fit_file(self, path, block_chars=50_000_000, min_word_freq=1):
        """Count word frequencies by streaming the corpus file in blocks, then learn the merges."""
        t0 = time.time()
        word_freqs = Counter()
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()          # the last word may be cut in half: finish it with the next block
                else:
                    leftover = ""
                word_freqs.update(parts)
        if leftover:
            word_freqs[leftover] += 1
        print(f"  counted {sum(word_freqs.values()):,} words ({len(word_freqs):,} distinct) in {time.time() - t0:.0f}s", flush=True)
        return self.fit_word_freqs(word_freqs, min_word_freq)

    def fit_word_freqs(self, word_freqs, min_word_freq=1):
        t0 = time.time()
        target = self.vocab_size
        words = [w for w, c in word_freqs.items() if c >= min_word_freq]
        freqs = [word_freqs[w] for w in words]
        splits = [self._word_symbols(w) for w in words]
        del word_freqs

        base = set()
        for syms in splits:
            base.update(syms)
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.merges = []
        self.itos.extend(sorted(base))
        print(f"  base byte alphabet: {len(base)} symbols; learning merges from {len(words):,} word types", flush=True)

        # pair_counts: how often each adjacent pair occurs (weighted by word frequency).
        # pair_words: which word types contain each pair, so a merge only revisits those words.
        pair_counts = {}
        pair_words = defaultdict(set)
        for i, syms in enumerate(splits):
            f = freqs[i]
            for j in range(len(syms) - 1):
                p = (syms[j], syms[j + 1])
                pair_counts[p] = pair_counts.get(p, 0) + f
                pair_words[p].add(i)

        # Max-heap of (-count, pair): pops the most frequent pair, ties -> lexicographically smallest pair
        # (identical to Assignment 1's min over (-count, pair)). Entries go stale when a count changes; a
        # fresh entry is pushed for every changed pair and stale ones are skipped when they reach the top.
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        while len(self.itos) < target:
            best, best_f = None, 0
            while heap:
                neg_c, p = heap[0]
                if pair_counts.get(p, 0) == -neg_c:
                    best, best_f = p, -neg_c
                    heapq.heappop(heap)
                    break
                heapq.heappop(heap)                  # stale entry
            if best is None:
                break
            if best_f < 2:
                print("  stopping: no pair occurs more than once", flush=True)
                break

            self.merges.append([best[0], best[1]])
            self.itos.append(best[0] + best[1])

            changed = set()
            for i in list(pair_words[best]):
                syms, f = splits[i], freqs[i]
                for j in range(len(syms) - 1):
                    p = (syms[j], syms[j + 1])
                    c = pair_counts.get(p, 0) - f
                    if c <= 0:
                        pair_counts.pop(p, None)
                    else:
                        pair_counts[p] = c
                    pair_words[p].discard(i)
                    changed.add(p)
                new_syms = self._merge(syms, best)
                splits[i] = new_syms
                for j in range(len(new_syms) - 1):
                    p = (new_syms[j], new_syms[j + 1])
                    pair_counts[p] = pair_counts.get(p, 0) + f
                    pair_words[p].add(i)
                    changed.add(p)
            for p in changed:
                c = pair_counts.get(p)
                if c:
                    heapq.heappush(heap, (-c, p))
            del pair_words[best]

            if len(self.merges) % 1000 == 0:
                print(f"  merges {len(self.merges):,} | vocab {len(self.itos):,} | {time.time() - t0:.0f}s", flush=True)

        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self._ranks = {(a, b): r for r, (a, b) in enumerate(self.merges)}
        self._id_cache = {}
        self.vocab_size = len(self.itos)      # the size the embedding table must actually have
        return self

    # ---- save / load (so the tokenizer is trained once and then reused) ----
    def save(self, path, meta=None):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges, "itos": self.itos,
                       "meta": meta or {}}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(data["vocab_size"])
        tok.merges = [list(m) for m in data["merges"]]
        tok.itos = list(data["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok._ranks = {(a, b): r for r, (a, b) in enumerate(tok.merges)}
        tok._id_cache = {}
        tok.vocab_size = len(tok.itos)
        return tok

    # ---- inference ----
    def _tokenize_word(self, word):
        syms = self._word_symbols(word)
        while len(syms) > 1:
            best_rank, best_j = None, -1
            for j in range(len(syms) - 1):
                r = self._ranks.get((syms[j], syms[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_j = r, j
            if best_rank is None:
                break
            syms = self._merge(syms, (syms[best_j], syms[best_j + 1]))
        return syms

    def _word_ids(self, word):
        ids = self._id_cache.get(word)
        if ids is None:
            unk = self.stoi["<unk>"]
            ids = [self.stoi.get(t, unk) for t in self._tokenize_word(word)]     # symbols never seen in training -> <unk>
            self._id_cache[word] = ids
        return ids

    def tokenize(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._tokenize_word(word))
        return out

    def get_vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._word_ids(word))
        return out

    def encode_file(self, path, block_chars=5_000_000):
        """Encode a whole text file into a packed array of ids, block by block (bounded memory)."""
        out = array.array("i")
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()
                else:
                    leftover = ""
                cache = self._id_cache
                for word in parts:
                    ids = cache.get(word)
                    if ids is None:
                        ids = self._word_ids(word)
                    out.extend(ids)
        if leftover:
            out.extend(self._word_ids(leftover))
        return out

    def decode(self, ids):
        """ids -> string. Words are re-joined with single spaces; <unk> decodes to '?'."""
        words, cur = [], bytearray()
        for i in ids:
            i = int(i)
            if not 0 <= i < len(self.itos):
                continue
            tok = self.itos[i]
            if tok in ("<pad>", "<s>", "</s>"):
                continue
            if tok == "<unk>":
                cur.extend(b"?")
                continue
            ends_word = tok.endswith(_END)
            if ends_word:
                tok = tok[: -len(_END)]
            cur.extend(ord(c) & 0xFF for c in tok)
            if ends_word:
                words.append(bytes(cur).decode("utf-8", errors="replace"))
                cur = bytearray()
        if cur:
            words.append(bytes(cur).decode("utf-8", errors="replace"))
        return " ".join(words)



bpe_tokenizer = BPETokenizer.load(fetch_model_file(TOKENIZER_NAME))
VOCAB = bpe_tokenizer.get_vocab_size()
print("tokenizer:", TOKENIZER_NAME, "| vocab", VOCAB)

(…)full_v40000_160000000w_seed42_minf1.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

(found a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json in the old repo kavincg23/cs781-checkpoints)
tokenizer: a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json | vocab 40000


In [3]:
import numpy as np, re
P5 = "Phase_5_LSI"
def load_json(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)
RAW = {sp: load_json(get_dataset_file(f"{P5}/{d}/{sp}.json")) for sp, d in (("train", "Train"), ("dev", "Train"), ("test", "Test"))}
for sp, rows in RAW.items():
    print(f"{sp}: {type(rows).__name__} with {len(rows)} records; fields of the first record: {list(rows[0].keys()) if isinstance(rows, list) else 'dict'}")
FIELDS = list(RAW["train"][0].keys())
ID_KEY = next((k for k in FIELDS if k.lower() in ("id", "case_id", "caseid")), FIELDS[0])
LABEL_KEY = next((k for k in FIELDS if k.lower() in ("labels", "label", "statutes", "statute_ids", "y")), None)
print("id field:", ID_KEY, "| label field:", LABEL_KEY, "| text field type:", type(RAW["train"][0]["text"]).__name__,
      "| label value type:", type(RAW["train"][0][LABEL_KEY]).__name__, "| example labels:", RAW["train"][0][LABEL_KEY])
other = [k for k in FIELDS if k not in (ID_KEY, LABEL_KEY, "text")]
print("other fields present in train:", other, "| in test:", [k for k in RAW["test"][0].keys() if k not in (ID_KEY, "text")])
def text_of(r):
    t = r["text"]
    return " ".join(t) if isinstance(t, list) else str(t)
def labs(r):
    v = r[LABEL_KEY]
    return sorted(set(int(x) for x in (v if isinstance(v, (list, tuple)) else [v])))

# ---- 1. ids, duplicates, overlaps (the Phase-4 data had dev/test overlap: check again) ----
ids = {sp: [r[ID_KEY] for r in RAW[sp]] for sp in RAW}
for sp in ids:
    print(f"{sp}: {len(ids[sp])} ids, {len(set(ids[sp]))} unique")
print("id overlap train/dev:", len(set(ids["train"]) & set(ids["dev"])), "| train/test:", len(set(ids["train"]) & set(ids["test"])), "| dev/test:", len(set(ids["dev"]) & set(ids["test"])))
H = lambda t: hash(t[:2000] + t[-2000:])
hs = {sp: {H(text_of(r)) for r in RAW[sp]} for sp in RAW}
print("same text (first+last 2000 chars) train/dev:", len(hs["train"] & hs["dev"]), "| train/test:", len(hs["train"] & hs["test"]), "| dev/test:", len(hs["dev"] & hs["test"]))
for sp in ("train", "dev"):
    seen, conflict, exact = {}, 0, 0
    for r in RAW[sp]:
        k = r[ID_KEY]; sig = (hash(text_of(r)), tuple(labs(r)))
        if k in seen:
            if seen[k] == sig: exact += 1
            else: conflict += 1
        else: seen[k] = sig
    print(f"{sp}: exact repeated records {exact}, same id but different text/labels {conflict}")

# ---- 2. labels ----
LB = {sp: [labs(r) for r in RAW[sp]] for sp in ("train", "dev")}
cnt = {sp: Counter(l for ls in LB[sp] for l in ls) for sp in LB}
allL = sorted(set(cnt["train"]) | set(cnt["dev"]))
print(f"\nlabel ids: train {len(cnt['train'])} distinct, dev {len(cnt['dev'])} distinct, union {len(allL)}; min {allL[0]}, max {allL[-1]} -> {'contiguous 0..L-1' if allL == list(range(len(allL))) else 'NOT contiguous'}")
print("labels in dev but never in train:", sorted(set(cnt["dev"]) - set(cnt["train"]))[:30], "| count:", len(set(cnt["dev"]) - set(cnt["train"])))
print("labels in train but never in dev:", len(set(cnt["train"]) - set(cnt["dev"])))
for sp in LB:
    k = np.array([len(l) for l in LB[sp]])
    print(f"{sp}: labels per document mean {k.mean():.2f}, median {np.median(k):.0f}, p90 {np.percentile(k, 90):.0f}, p99 {np.percentile(k, 99):.0f}, max {k.max()}; documents with 0 labels {int((k == 0).sum())}, exactly 1 label {int((k == 1).sum())} ({(k == 1).mean():.3f})")
sup = np.array([cnt["train"].get(l, 0) for l in allL])
bins = [(1, 1), (2, 4), (5, 9), (10, 49), (50, 199), (200, 999), (1000, 10**9)]
print("train support per label (number of labels with that many training documents): " + ", ".join(f"{a}-{b if b < 10**9 else 'inf'}: {int(((sup >= a) & (sup <= b)).sum())}" for a, b in bins) + f", zero: {int((sup == 0).sum())}")
dsup = np.array([cnt["dev"].get(l, 0) for l in allL])
print("dev support per label: " + ", ".join(f"{a}-{b if b < 10**9 else 'inf'}: {int(((dsup >= a) & (dsup <= b)).sum())}" for a, b in bins) + f", zero: {int((dsup == 0).sum())}")
print("most frequent labels (train count / dev count):", [(l, cnt['train'][l], cnt['dev'].get(l, 0)) for l, _ in cnt['train'].most_common(15)])
print("share of all train label occurrences covered by the top 10 / 50 / 100 labels:", [round(sum(c for _, c in cnt['train'].most_common(n)) / sum(cnt['train'].values()), 3) for n in (10, 50, 100)])
sets = Counter(tuple(l) for l in LB["train"])
dev_seen = np.mean([tuple(l) in sets for l in LB["dev"]])
print(f"distinct label sets in train: {len(sets)} (most common: {sets.most_common(5)}); share of dev documents whose exact label set also occurs in train: {dev_seen:.3f}")
# co-occurrence: pairs with support >= 20, by Jaccard
pair = Counter()
for l in LB["train"]:
    for i in range(len(l)):
        for j in range(i + 1, len(l)):
            pair[(l[i], l[j])] += 1
jac = sorted(((c / (cnt['train'][a] + cnt['train'][b] - c), a, b, c) for (a, b), c in pair.items() if c >= 20), reverse=True)[:10]
print("label pairs that co-occur most strongly (Jaccard, a, b, joint count):", [(round(j, 2), a, b, c) for j, a, b, c in jac])

# ---- 3. lengths ----
rng = random.Random(0)
def pct(a, ps=(5, 25, 50, 75, 90, 95, 99, 100)):
    a = sorted(a); return {p: a[min(len(a) - 1, int(p / 100 * len(a)))] for p in ps}
TOKS = {}
for sp in RAW:
    words = [len(text_of(r).split()) for r in RAW[sp]]
    sample = rng.sample(RAW[sp], min(1500, len(RAW[sp])))
    toks = [len(bpe_tokenizer.encode(text_of(r))) for r in sample]; TOKS[sp] = toks
    print(f"\n{sp}: words {pct(words)}\n   BPE tokens (sample {len(sample)}) {pct(toks)}  tokens/word {np.mean(toks) / max(1.0, np.mean([len(text_of(r).split()) for r in sample])):.2f}")
    print("   share of documents with <= N tokens: " + ", ".join(f"{cap}: {np.mean(np.array(toks) <= cap):.3f}" for cap in (512, 1024, 2048, 4096, 8192, 16384)))
tr = RAW["train"]
tk = np.array([len(text_of(r).split()) for r in tr]); nl = np.array([len(l) for l in LB["train"]])
print(f"\ncorrelation between document length (words) and number of labels (train): {np.corrcoef(tk, nl)[0, 1]:.3f}")

# ---- 4. statute mentions in the text: are the labels recoverable from section / act mentions, and where in the document are they? ----
SEC = re.compile(r"\b[Ss]ections?\s+(\d+[A-Za-z]{0,2})\b")
ACT = re.compile(r"((?:[A-Z][A-Za-z\(\)\.&']*\s+){1,6}Act(?:,?\s+(?:of\s+)?\d{4})?)")
_si = sorted(rng.sample(range(len(tr)), min(3000, len(tr))))      # sample: the regexes are slow on 379 MB of text
trs = [tr[i] for i in _si]; LBs = [LB["train"][i] for i in _si]
print(f"(section 4 uses a random sample of {len(trs)} training documents)")
sec_docs = [set(SEC.findall(text_of(r))) for r in trs]
act_docs = [set(a.strip() for a in ACT.findall(text_of(r))) for r in trs]
print(f"documents mentioning at least one 'Section N': {np.mean([len(s) > 0 for s in sec_docs]):.3f}; at least one '... Act': {np.mean([len(s) > 0 for s in act_docs]):.3f}")
print("most frequent acts:", Counter(a for s in act_docs for a in s).most_common(10))
print("most frequent section numbers:", Counter(a for s in sec_docs for a in s).most_common(10))
doc_terms = [{("sec " + x) for x in s} | {("act " + x) for x in a} for s, a in zip(sec_docs, act_docs)]
tdf = Counter(t for ts in doc_terms for t in ts)
print("\nfor the 10 most frequent labels: the mention that best predicts the label (precision of 'mention present -> label', recall, document count)")
for l, c in cnt["train"].most_common(10):
    best = []
    c = sum(1 for ls in LBs if l in ls)
    pos = [i for i, ls in enumerate(LBs) if l in ls]
    pc = Counter(t for i in pos for t in doc_terms[i])
    for t, k in pc.items():
        if c and k >= max(5, 0.05 * c):
            best.append((k / tdf[t], k / c, t, tdf[t]))
    best.sort(reverse=True)
    print(f"  label {l} (docs in sample {c}): {[(round(p, 2), round(r, 2), t, n) for p, r, t, n in best[:3]]}")
# where in the document do 'Section N' mentions sit (relative position of all mentions)?
pos_all = []
for r in rng.sample(trs, min(600, len(trs))):
    t = text_of(r); L = max(1, len(t))
    pos_all += [m.start() / L for m in SEC.finditer(t)]
if pos_all:
    h = np.histogram(pos_all, bins=10, range=(0, 1))[0]
    print("\nshare of 'Section N' mentions by decile of the document (0-10% ... 90-100%):", [round(float(x / h.sum()), 3) for x in h])

# ---- 5. trivial baselines on dev (to know what 'nothing learned' scores; the evaluator derives the label dimension from the reference data) ----
def f1_scores(Y, P):
    tp = (Y & P).sum(0); fp = ((~Y) & P).sum(0); fn = (Y & (~P)).sum(0)
    f = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
    micro = 2 * tp.sum() / max(1, 2 * tp.sum() + fp.sum() + fn.sum())
    return float(f.mean()), float(micro), float((Y == P).all(1).mean())
L = len(allL); idx = {l: i for i, l in enumerate(allL)}
Y = np.zeros((len(LB["dev"]), L), bool)
for i, ls in enumerate(LB["dev"]):
    for l in ls: Y[i, idx[l]] = True
print(f"\nbaselines on dev ({len(Y)} documents, {L} labels; macro-F1 / micro-F1 / subset accuracy; labels without support count as F1 0):")
for k in (1, 3, 5):
    P = np.zeros_like(Y); 
    for l, _ in cnt["train"].most_common(k): P[:, idx[l]] = True
    print(f"  always predict the {k} most frequent train labels: {tuple(round(x, 4) for x in f1_scores(Y, P))}")
print(f"  labels with at least one dev document: {int((Y.sum(0) > 0).sum())} of {L} (macro-F1 averages over the evaluator's label dimension; unsupported labels cannot score)")

# ---- 6. two documents ----
for r in rng.sample(tr, 2):
    t = text_of(r)
    print(f"\n--- id {r[ID_KEY]}, labels {labs(r)}, {len(t.split())} words ---\nSTART: {t[:400]!r}\nEND:   {t[-400:]!r}")
SUMMARY = dict(fields=FIELDS, label_key=LABEL_KEY, id_key=ID_KEY, n_labels=L, sizes={sp: len(v) for sp, v in RAW.items()},
               label_support_train={int(l): cnt["train"].get(l, 0) for l in allL}, tokens_sample={sp: pct(v) for sp, v in TOKS.items()})
with open(f"{WORK}/p5_data_inspection.json", "w") as f:
    json.dump(SUMMARY, f, indent=1, default=str)
print("\nSaved", f"{WORK}/p5_data_inspection.json", "- send me this whole printout.")

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

train: list with 42750 records; fields of the first record: ['id', 'text', 'labels']
dev: list with 10181 records; fields of the first record: ['id', 'text', 'labels']
test: list with 13019 records; fields of the first record: ['id', 'text']
id field: id | label field: labels | text field type: str | label value type: list | example labels: [69, 9, 3]
other fields present in train: [] | in test: []
train: 42750 ids, 42750 unique
dev: 10181 ids, 10181 unique
test: 13019 ids, 13019 unique
id overlap train/dev: 0 | train/test: 0 | dev/test: 0
same text (first+last 2000 chars) train/dev: 90 | train/test: 117 | dev/test: 67
train: exact repeated records 0, same id but different text/labels 0
dev: exact repeated records 0, same id but different text/labels 0

label ids: train 100 distinct, dev 100 distinct, union 100; min 0, max 99 -> contiguous 0..L-1
labels in dev but never in train: [] | count: 0
labels in train but never in dev: 0
train: labels per document mean 3.70, median 3, p90 7, p9